<a href="https://colab.research.google.com/github/nur21horin/machine_learning/blob/main/neuclear_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('/content/Nuclear_Power_Plant_CPS_Dataset.csv')

In [4]:
df.sample()

,Timestamp,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Pump Status,Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m),Anomaly Detected
1189,2024-10-17 05:49:00,335.474833,485.52874,15.801885,0.548923,3121.202664,ON,952.67546,75.372649,205.801532,2.678053,5.237312,No


In [5]:
df.describe()

,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m)
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,350.575733,470.369775,16.008382,0.581900,3192.985499,1000.442858,70.123696,219.841024,1.990574,5.198914
std,14.825930,14.756203,0.511526,0.295586,201.808719,52.127700,4.996608,14.363382,0.500757,0.098022
min,301.380990,424.707318,14.504432,-0.353011,2620.097224,815.581735,50.816722,161.163996,0.312210,4.896601
25%,340.624931,460.370079,15.646462,0.386899,3061.261439,965.441255,66.845331,210.077114,1.661971,5.135156
50%,350.724549,470.184693,16.008505,0.593313,3192.707942,999.437283,70.100292,220.139668,1.975637,5.199864
75%,360.140211,480.089684,16.346386,0.777762,3326.666520,1036.376938,73.409052,229.360633,2.318087,5.264305
max,407.790972,517.896614,17.963119,1.572928,3822.582040,1155.884056,86.886915,263.711638,3.688884,5.528412


In [9]:
df.isnull().sum()

,0
Timestamp,0
Reactor Temp (°C),0
Coolant Flow Rate (L/s),0
Pressure (MPa),0
Radiation Level (μSv/h),0
Turbine Speed (RPM),0
Pump Status,0
Power Output (MW),0
Control Rod Position (%),0
Steam Flow Rate (kg/s),0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Timestamp                 1200 non-null   object 
 1   Reactor Temp (°C)         1200 non-null   float64
 2   Coolant Flow Rate (L/s)   1200 non-null   float64
 3   Pressure (MPa)            1200 non-null   float64
 4   Radiation Level (μSv/h)   1200 non-null   float64
 5   Turbine Speed (RPM)       1200 non-null   float64
 6   Pump Status               1200 non-null   object 
 7   Power Output (MW)         1200 non-null   float64
 8   Control Rod Position (%)  1200 non-null   float64
 9   Steam Flow Rate (kg/s)    1200 non-null   float64
 10  Vibration Level (mm/s)    1200 non-null   float64
 11  Water Level (m)           1200 non-null   float64
 12  Anomaly Detected          1200 non-null   object 
dtypes: float64(10), object(3)
memory usage: 122.0+ KB


In [10]:
df=df.drop(columns=["Timestamp"])

In [15]:
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,IsolationForest
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
#Encode categorical Pump Status
df["Pump Status"]=LabelEncoder().fit_transform(df["Pump Status"])
# Encode target variable (Anomaly detected)
df["Anomaly Detected"]=LabelEncoder().fit_transform(df["Anomaly Detected"])
# Separate features and target
X=df.drop(columns=["Anomaly Detected"])
y=df["Anomaly Detected"]
# Scale features
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [17]:
rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_scaled,y)


RandomForestClassifier(random_state=42)

In [18]:
importance=pd.Series(rf.feature_importances_,index=X.columns)
print(importance.sort_values(ascending=False))

Reactor Temp (°C)           0.110047
Coolant Flow Rate (L/s)     0.102369
Turbine Speed (RPM)         0.099449
Vibration Level (mm/s)      0.098666
Steam Flow Rate (kg/s)      0.098124
Control Rod Position (%)    0.098065
Water Level (m)             0.096503
Pressure (MPa)              0.095897
Radiation Level (μSv/h)     0.095714
Power Output (MW)           0.093378
Pump Status                 0.011787
dtype: float64


In [22]:
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.2,random_state=42)
rf_model=RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)
y_pred=rf_model.predict(X_test)

print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[236   0]
 [  4   0]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       236
           1       0.00      0.00      0.00         4

    accuracy                           0.98       240
   macro avg       0.49      0.50      0.50       240
weighted avg       0.97      0.98      0.98       240



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
iso=IsolationForest(contamination=0.05,random_state=42)
iso.fit(X_scaled)

# prediction anomalies

anomaly_pred=iso.predict(X_scaled)

df["IsoForest_Anomaly"]=np.where(anomaly_pred==-1,1,0)

print(df[["Radiation Level (μSv/h)", "Reactor Temp (°C)", "IsoForest_Anomaly"]].head(20))

    Radiation Level (μSv/h)  Reactor Temp (°C)  IsoForest_Anomaly
0                  0.352819         357.450712                  0
1                  0.384267         347.926035                  0
2                  1.214944         359.715328                  0
3                  0.600205         372.845448                  0
4                  0.834639         346.487699                  0
5                  0.362730         346.487946                  0
6                  0.366044         373.688192                  0
7                  0.924595         361.511521                  0
8                  0.195127         342.957884                  0
9                  0.475436         358.138401                  0
10                 0.610208         343.048735                  0
11                 0.633412         343.014054                  0
12                 0.546272         353.629434                  0
13                 0.718746         321.300796                  0
14        

In [16]:
df.corr(numeric_only=True)

,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Pump Status,Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m),Anomaly Detected
Reactor Temp (°C),1.000000,0.015838,0.005271,0.031829,-0.023609,-0.015193,-0.034660,-0.022849,-0.009548,0.001287,-0.008502,-0.024976
Coolant Flow Rate (L/s),0.015838,1.000000,0.017964,0.024600,-0.007943,0.053233,0.004533,-0.004523,0.004688,0.001499,0.055470,0.009414
Pressure (MPa),0.005271,0.017964,1.000000,-0.031210,0.041787,-0.033178,-0.006501,-0.010827,0.029230,-0.012847,0.015695,-0.055480
Radiation Level (μSv/h),0.031829,0.024600,-0.031210,1.000000,0.047175,-0.035087,-0.009878,-0.006436,-0.009912,0.039334,0.026763,-0.020171
Turbine Speed (RPM),-0.023609,-0.007943,0.041787,0.047175,1.000000,0.006944,0.028796,0.005255,-0.031148,-0.012294,0.012113,0.000723
Pump Status,-0.015193,0.053233,-0.033178,-0.035087,0.006944,1.000000,-0.012793,-0.007359,0.053688,0.024692,-0.012200,-0.026721
Power Output (MW),-0.034660,0.004533,-0.006501,-0.009878,0.028796,-0.012793,1.000000,-0.035470,0.014457,-0.029645,0.002803,0.042300
Control Rod Position (%),-0.022849,-0.004523,-0.010827,-0.006436,0.005255,-0.007359,-0.035470,1.000000,0.031892,0.020284,-0.065356,-0.025635
Steam Flow Rate (kg/s),-0.009548,0.004688,0.029230,-0.009912,-0.031148,0.053688,0.014457,0.031892,1.000000,0.022603,-0.007625,0.033652
Vibration Level (mm/s),0.001287,0.001499,-0.012847,0.039334,-0.012294,0.024692,-0.029645,0.020284,0.022603,1.000000,0.006760,-0.001128
